# TAMER OCR v2.2 — Clean Orchestrator

This notebook is an execution-only wrapper. It purges the environment to ensure a clean state, pulls the latest remote codebase, authenticates, and hands control over to `Trainer.run()`.

## 1. Absolute Purge (Clean Slate Protocol)

In [ ]:
import os
import shutil
import gc

def purge_environment():
    print("☣️ INITIATING CLEAN SLATE PROTOCOL...")
    
    directories_to_nuke = [
        '/content/AI_PROJECT_TAMER_Complete',
        '/kaggle/working/AI_PROJECT_TAMER_Complete',
        '/content/data',
        '/kaggle/working/data',
        '/content/outputs',
        '/kaggle/working/outputs',
        '/content/checkpoints',
        '/kaggle/working/checkpoints',
        '/content/logs',
        '/kaggle/working/logs',
        '/root/.cache/huggingface/datasets'
    ]
    
    for directory in directories_to_nuke:
        if os.path.exists(directory):
            print(f"  [DELETING] {directory}")
            shutil.rmtree(directory, ignore_errors=True)
            
    # Verify deletion
    for directory in directories_to_nuke:
        assert not os.path.exists(directory), f"Failed to delete {directory}!"
            
    gc.collect()
    print("✅ Environment successfully purged. Ready for deterministic build.")

purge_environment()

## 2. Environment Setup & Authentication

In [ ]:
import os
import getpass

# Detect Environment
IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.chdir(WORK_DIR)

print(f"🌍 Environment: {'Kaggle' if IS_KAGGLE else 'Colab'}")
print(f"📂 Working Directory: {WORK_DIR}\n")

print("🔑 AUTHENTICATION")
# HuggingFace Token
hf_token = os.getenv('HF_TOKEN', '')
if not hf_token:
    hf_token = getpass.getpass('Enter HuggingFace Token (WRITE access needed): ')
os.environ['HF_TOKEN'] = hf_token

# Kaggle Credentials
kaggle_username = os.getenv('KAGGLE_USERNAME', '')
if not kaggle_username:
    kaggle_username = input('Enter Kaggle Username: ').strip()
    
kaggle_key = os.getenv('KAGGLE_KEY', '')
if not kaggle_key:
    kaggle_key = getpass.getpass('Enter Kaggle API Key: ')
    
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Configure Kaggle CLI Auth File
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print("✅ Authentication configured.")

## 3. Dependencies & Codebase Retrieval

In [ ]:
print("📦 Installing dependencies...")
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless
print("✅ Dependencies installed.\n")

print("📥 Cloning Repository...")
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
!git clone {REPO_URL}
print("✅ Codebase retrieved.")

## 4. System Path Injection & Configuration
Inject the codebase into the python path and configure the environment-specific directories dynamically.

In [ ]:
import sys

REPO_ROOT = os.path.join(WORK_DIR, 'AI_PROJECT_TAMER_Complete')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f"🔗 System Path configured: {REPO_ROOT}")

from tamer_ocr.config import Config
from huggingface_hub import HfApi

# Initialize Configuration
config = Config()

# Dynamic Directory Mapping
config.data_dir = os.path.join(WORK_DIR, 'data')
config.output_dir = os.path.join(WORK_DIR, 'outputs')
config.checkpoint_dir = os.path.join(WORK_DIR, 'checkpoints')
config.log_dir = os.path.join(WORK_DIR, 'logs')

for d in [config.data_dir, config.output_dir, config.checkpoint_dir, config.log_dir]:
    os.makedirs(d, exist_ok=True)

# HuggingFace Hub Setup
hf_api = HfApi(token=os.environ['HF_TOKEN'])
hf_username = hf_api.whoami()['name']
config.hf_token = os.environ['HF_TOKEN']
config.hf_repo_id = f"{hf_username}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_username}/tamer-preprocessed"

# Training Hyperparameters
config.batch_size = 8
config.accumulation_steps = 4  # Effective batch = 32
config.num_epochs = 150
config.encoder_lr = 1e-5
config.decoder_lr = 1e-4
config.label_smoothing = 0.1
config.total_training_steps = 50000

# Checkpoint Safety
config.checkpoint_every_epochs = 3
config.keep_last_n_checkpoints = 3

print("✅ Configuration fully initialized and synchronized.")

## 5. Execute Main Pipeline (Strict Mode)

In [ ]:
from tamer_ocr.core.trainer import Trainer

print("🚀 Launching TAMER OCR Pipeline...")
trainer = Trainer(config)
trainer.run()  # Fully self-contained execution loop

## 6. Final Output Synchronization & Hub Push

In [ ]:
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

best_model_path = os.path.join(config.checkpoint_dir, 'best.pt')
if os.path.exists(best_model_path):
    print(f"📤 Pushing best model to {config.hf_repo_id}...")
    push_checkpoint_to_hf(best_model_path, config, epoch=0, is_best=True)
    print("✅ Push complete.")
else:
    print("⚠️ No best.pt found. The model may not have finished the first evaluation cycle.")